# Solution to p07

The class parameter is expanded as a view along the batch dimension, so all examples use the same learned parameter without data-dependent replication. Positions are added only after projection and concatenation.

In [ ]:
import torch
from torch import nn

torch.manual_seed(20260901)
patch_tokens = torch.tensor([
    [[0.,1.,4.,5.], [2.,3.,6.,7.], [8.,9.,12.,13.], [10.,11.,14.,15.]],
    [[16.,17.,20.,21.], [18.,19.,22.,23.], [24.,25.,28.,29.], [26.,27.,30.,31.]],
], dtype=torch.float32)
projection = nn.Linear(4, 8, bias=True)
with torch.no_grad():
    projection.weight.copy_(torch.arange(32, dtype=torch.float32).reshape(8, 4) / 100.0)
    projection.bias.copy_(torch.tensor([0., .01, .02, .03, .04, .05, .06, .07]))
cls_token = nn.Parameter(torch.tensor([[[.1,.2,.3,.4,.5,.6,.7,.8]]], dtype=torch.float32))
positions = nn.Parameter(torch.zeros((1, 5, 8), dtype=torch.float32))
with torch.no_grad():
    positions.copy_(torch.arange(40, dtype=torch.float32).reshape(1, 5, 8) / 100.0)
ATOL = 1e-6
RTOL = 1e-6

def make_vit_input(patch_tokens, projection, cls_token, positions):
    if not isinstance(patch_tokens, torch.Tensor) or patch_tokens.shape != (2,4,4):
        raise ValueError("patch_tokens must have shape (2,4,4)")
    if patch_tokens.dtype != torch.float32 or patch_tokens.device.type != "cpu":
        raise ValueError("patch_tokens must be CPU float32")
    if not isinstance(projection, nn.Linear) or projection.in_features != 4 or projection.out_features != 8:
        raise ValueError("projection must be Linear(4,8)")
    if cls_token.shape != (1,1,8) or positions.shape != (1,5,8):
        raise ValueError("class/position shape mismatch")
    if cls_token.dtype != torch.float32 or positions.dtype != torch.float32:
        raise ValueError("class and positions must be float32")
    projected = projection(patch_tokens)
    expanded_cls = cls_token.expand(patch_tokens.shape[0], -1, -1)
    positioned = torch.cat((expanded_cls, projected), dim=1) + positions
    return positioned, expanded_cls

vit_input, expanded_cls = make_vit_input(patch_tokens, projection, cls_token, positions)


### Answer check

In [ ]:
expected = torch.tensor([
 [[.10,.21,.32,.43,.54,.65,.76,.87],[.32,.74,1.16,1.58,2,2.42,2.84,3.26],[.52,1.26,2,2.74,3.48,4.22,4.96,5.70],[.96,2.66,4.36,6.06,7.76,9.46,11.16,12.86],[1.16,3.18,5.20,7.22,9.24,11.26,13.28,15.30]],
 [[.10,.21,.32,.43,.54,.65,.76,.87],[1.28,4.26,7.24,10.22,13.20,16.18,19.16,22.14],[1.48,4.78,8.08,11.38,14.68,17.98,21.28,24.58],[1.92,6.18,10.44,14.70,18.96,23.22,27.48,31.74],[2.12,6.70,11.28,15.86,20.44,25.02,29.60,34.18]],
], dtype=torch.float32)
assert vit_input.shape == (2,5,8) and expanded_cls.shape == (2,1,8)
assert expanded_cls.stride(0) == 0
torch.testing.assert_close(vit_input, expected, atol=ATOL, rtol=RTOL)
torch.testing.assert_close(expanded_cls[0], cls_token[0], atol=ATOL, rtol=RTOL)
